In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import IntervalTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser
import matplotlib.pyplot as plt

### Task Incremental Learning

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Train the model on the original task.

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Train subsequent tasks with bounds.

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, context_id=i)
    interval_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test, context_id=i)

### Domain Incremental Learning

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.9,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    batch_size=400,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, epochs=3)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)

### Class Incremental Learning

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    n_certificate_samples=400,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, epochs=3, batch_size=512)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)

### Class Incremental Learning + Regulariser

In [136]:
config = {
    "seed": 0,
    "n_iters": 200,
    "n_epochs": 5,
    "batchsize": 64,
    "l2_lambda": 0.01,
    "obj_alpha": 0,
    "checkpoint": 20,
    "hidden_dim": 128,
    "weight_decay": 0,
    "learning_rate": 0.027,
    "min_acc_limit": 1,
    "unbias_lambda": 0.01,
    "min_acc_increment": 0.12,
    "dual_learning_rate": 0.01,
    "rashomon_batchsize": 400,
    "penalty_coefficient": 1,
    "projection_strategy": "sample_largest_closest",
    "primal_learning_rate": 0.33,
}

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks(
    seed=config["seed"], train_val_split_ratio=1
)

model = models.get_mnist_model(device="cuda", seed=config["seed"])

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [138]:
unbias = UnbiasRegulariser(lmbd=config["unbias_lambda"])
l2 = L2Regulariser(lmbd=config["l2_lambda"])
regulariser = MultiRegulariser([l2, unbias])

trainer = SimpleTrainer(model, seed=None)
trainer.train(
    train_tasks[0],
    test_tasks[0],
    epochs=config["n_epochs"],
    batch_size=config["batchsize"],
    regulariser=regulariser,
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████████████| 5/5 [00:09<00:00,  1.89s/it, train_loss=0.0798, val_loss=0.0191, val_acc=0.991]


Test Results: [(0.0231, 0.993)]


[(0.0231, 0.993)]

In [ ]:
checkpoint = (
    config["n_iters"] // config["n_checkpoints"]
    if config.get("n_checkpoints")
    else config["checkpoint"]
)
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=checkpoint,
    n_iters=config["n_iters"],
    min_acc_limit=config["min_acc_limit"],
    min_acc_increment=config["min_acc_increment"],
    primal_learning_rate=config["primal_learning_rate"],
    dual_learning_rate=config["dual_learning_rate"],
    n_certificate_samples=config["rashomon_batchsize"],
    penalty_coefficient=config["penalty_coefficient"],
    projection_strategy=config["projection_strategy"],
    paradigm="CIL",
    seed=None,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, test) in enumerate(zip(train_tasks[1:], test_tasks[1:]), start=1):
    interval_trainer.train(
        train,
        test,
        batch_size=config["batchsize"],
        epochs=config["n_epochs"],
        regulariser=regulariser,
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 1:
        interval_trainer.compute_rashomon_set(test)

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1289 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.86
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=0.99


100%|█████████████████████████████████████| 200/200 [00:08<00:00, 24.93it/s, size=1093.94, obj=0.177, min_soft_acc=1.000]


Final bbox:  Obj=0.18,  Size=1093.94,  Min acc hard=0.85,  Min acc soft=0.84
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['37.80', '163.47', '387.49', '564.84', '704.50', '850.60', '933.94', '1004.32', '1062.46', '1093.94']
Checkpoint certificates: ['0.92', '0.93', '0.84', '0.85', '0.88', '0.83', '0.85', '0.85', '0.85', '0.85']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|████████████████████████████| 5/5 [00:14<00:00,  2.84s/it, val_loss=0.7371, val_acc=0.8039, proj=9]


Test Results: [(0.3763, 0.8776), (0.535, 0.8044)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1288 (Positive = violated)
Computing Rashomon set within outer box of size: 1093.94
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.68
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.81,  Min acc soft=0.81


...

Test Results: [(0.3037, 0.9276), (0.8227, 0.7627), (0.9152, 0.7817), (1.8289, 0.3643)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1294 (Positive = violated)
Computing Rashomon set within outer box of size: 531.42
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.25
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.38,  Min acc soft=0.38


100%|██████████████████████████████████████| 200/200 [00:09<00:00, 21.29it/s, size=323.09, obj=0.052, min_soft_acc=0.231]


Final bbox:  Obj=0.05,  Size=323.09,  Min acc hard=0.21,  Min acc soft=0.21
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['11.64', '46.07', '100.41', '148.83', '189.22', '227.19', '261.82', '292.38', '311.61', '323.09']
Checkpoint certificates: ['0.26', '0.26', '0.25', '0.24', '0.24', '0.22', '0.22', '0.22', '0.21', '0.21']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|████████████████████████████| 5/5 [00:13<00:00,  2.70s/it, val_loss=2.6318, val_acc=0.1809, proj=9]


Test Results: [(0.3053, 0.9281), (0.8324, 0.7632), (0.8466, 0.7919), (2.5287, 0.2896), (2.4044, 0.1868)]
